**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Real-Time Signal Processing

The workshop [Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) promised: what changes when the signal *keeps coming* and every block has a **deadline**. Fixed-point arithmetic, block processing under a latency budget, and a simulated real-time pipeline with measured deadline misses — the glue between [DSP](./README.md), [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb), and [FPGA](../Intro_FPGA/README.md).

## 1. Pre-requisites

- [Filter Design](./Filter_Design.ipynb), [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) (block processing).
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — scheduling jitter is the enemy here.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
import time
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Fixed-Point Arithmetic* (~40 min)
**Goal:** represent signals in Q-format; measure quantization noise; watch overflow bite.
**Builds on:** [Intro to C](../Intro_Programming/Intro_C.ipynb) (bits). &nbsp; **Feeds into:** Session 2 (latency budgets).

---

## 2. Numbers Without a Float Unit

💡 **Intuition.** Microcontrollers and [FPGA fabric](../Intro_FPGA/Intro_FPGA.ipynb) do integer math. **Q-format** fakes fractions with an implicit binary point: Q1.15 stores $x \in [-1, 1)$ as $\mathrm{round}(x \cdot 2^{15})$ in an int16. Each quantization adds ~uniform noise of variance $\Delta^2/12$ — *6 dB of SNR per bit* — and every multiply must be re-scaled (>> 15) or the binary point drifts. The two failure modes to respect: **quantization noise** (graceful, hissy) and **overflow** (catastrophic, wrap-around).

In [2]:
def to_q15(x):  return np.clip(np.round(x * 2**15), -2**15, 2**15 - 1).astype(np.int16)
def from_q15(q): return q.astype(np.float64) / 2**15

t = np.arange(0, 1, 1/8000)
x = 0.7 * np.sin(2*np.pi*440*t)

for bits in [15, 11, 7]:
    scale = 2**bits
    q = np.round(x * scale) / scale
    snr = 10*np.log10(np.var(x) / np.var(x - q))
    print(f"Q1.{bits:2d}: measured SNR {snr:5.1f} dB   (rule of thumb ≈ 6.02×{bits}+1.76 = {6.02*bits+1.76:5.1f} dB)")

Q1.15: measured SNR  95.2 dB   (rule of thumb ≈ 6.02×15+1.76 =  92.1 dB)
Q1.11: measured SNR  71.4 dB   (rule of thumb ≈ 6.02×11+1.76 =  68.0 dB)
Q1. 7: measured SNR  46.9 dB   (rule of thumb ≈ 6.02×7+1.76 =  43.9 dB)


In [3]:
# A Q15 FIR filter, and the overflow trap
h = sig.firwin(31, 0.2)
h_q = to_q15(h)
x_q = to_q15(x)

# CORRECT: accumulate in int32 (headroom!), shift back once
acc = np.convolve(x_q.astype(np.int64), h_q.astype(np.int64))            # 32-bit safe products
y_good = from_q15(np.clip(acc >> 15, -2**15, 2**15 - 1).astype(np.int16))

# WRONG: no headroom — products wrap in int16
y_bad = np.zeros(len(x_q))
prod = (x_q.astype(np.int32)[:, None] * h_q.astype(np.int32)[None, :]) >> 15
prod16 = prod.astype(np.int16)                                            # silent wraparound!
for i_ in range(31, len(x_q)):
    y_bad[i_] = from_q15(np.sum(prod16[i_-30:i_+1, ::-1].diagonal()).astype(np.int16))

y_ref = np.convolve(x, h)[:len(x)]
plt.figure(figsize=(9, 2.6))
plt.plot(y_ref[400:600], "k--", linewidth=1, label="float reference")
plt.plot(y_good[400:600], label="Q15, int32 accumulator: indistinguishable")
plt.plot(y_bad[400:600], alpha=0.6, label="Q15, int16 accumulation: overflow chaos")
plt.legend(fontsize=8); plt.title("headroom is not optional")
plt.tight_layout(); plt.show()
print(f"error vs float — with headroom: {np.abs(y_good[:len(x)]-y_ref).max():.2e}   without: {np.abs(y_bad-y_ref).max():.2f}")

error vs float — with headroom: 5.52e-05   without: 0.68


/tmp/ipykernel_2703251/2941568280.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Latency Budgets & Block Processing* (~35 min)
**Goal:** count the milliseconds: block size sets the latency floor; compute must fit inside it.
**Builds on:** Session 1; [OS workshop](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb). &nbsp; **Feeds into:** Session 3 (a real-time pipeline).

---

## 3. The Budget

💡 **Intuition.** A real-time system processes block $k$ while block $k{+}1$ records. Two laws follow. **Latency floor:** you can't output before a block fills — latency ≥ one block (plus compute, plus output buffering); small blocks = low latency. **Throughput wall:** compute per block must finish in under one block-duration — or you fall behind *forever*. Small blocks also mean more per-block overhead ([OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) syscalls, scheduling), so the budget squeezes from both sides. Every audio interface's 'buffer size' knob is exactly this dial.

In [4]:
fs = 48_000
h_long = sig.firwin(2049, 0.1)                       # a deliberately heavy filter
x_blk = rng.standard_normal(fs)                      # 1 second of audio

print(f"{'block':>6} {'deadline':>9} {'direct conv':>12} {'FFT (overlap-save)':>19}")
for B in [64, 256, 1024, 4096]:
    deadline_ms = B / fs * 1000
    xb = x_blk[:B]
    tic = time.perf_counter()
    for _ in range(20): np.convolve(xb, h_long)
    t_direct = (time.perf_counter() - tic) / 20 * 1000
    tic = time.perf_counter()
    for _ in range(20): sig.oaconvolve(xb, h_long)
    t_fft = (time.perf_counter() - tic) / 20 * 1000
    verdict = lambda t: "✓" if t < deadline_ms else "✗ MISSES"
    print(f"{B:>6} {deadline_ms:>7.2f}ms {t_direct:>9.3f}ms {verdict(t_direct):8s} {t_fft:>9.3f}ms {verdict(t_fft)}")
print("\n→ the fast-convolution algorithms of [Foundations 1 S8] are what make small deadlines feasible")

 block  deadline  direct conv  FFT (overlap-save)
    64    1.33ms     0.015ms ✓            0.133ms ✓
   256    5.33ms     0.038ms ✓            0.052ms ✓
  1024   21.33ms     0.122ms ✓            0.055ms ✓
  4096   85.33ms     0.468ms ✓            0.084ms ✓

→ the fast-convolution algorithms of [Foundations 1 S8] are what make small deadlines feasible


---
### 🕐 Session 3 of 3 — *A Real-Time Pipeline, Simulated & Measured* (~40 min)
**Goal:** producer/consumer with deadlines; measure misses and jitter like an engineer.
**Builds on:** Session 2.

---

## 4. The Pipeline

Architecture (the [OS workshop's](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) producer/consumer, with a clock): an acquisition thread produces blocks on schedule; a processing thread must consume + filter each block before the next arrives. We measure the **headroom histogram** — the engineer's dashboard for 'will this survive a bad scheduling day?'

In [5]:
import threading, queue, collections

fs2, B = 48_000, 1024
block_dur = B / fs2
h_rt = sig.firwin(513, 0.1)
n_blocks = 300

q_ = queue.Queue(maxsize=8)
headroom, misses = [], 0
DONE = object()

def producer():
    next_t = time.perf_counter()
    for k in range(n_blocks):
        next_t += block_dur
        blk = rng.standard_normal(B)
        q_.put((blk, next_t))                      # deadline: before the NEXT block lands
        sleep = next_t - time.perf_counter()
        if sleep > 0: time.sleep(sleep)
    q_.put(DONE)

def consumer():
    global misses
    zi = np.zeros(len(h_rt) - 1)
    while (item := q_.get()) is not DONE:
        blk, deadline = item
        _, zi = sig.lfilter(h_rt, 1, blk, zi=zi)   # stateful streaming filter
        slack = deadline - time.perf_counter()
        headroom.append(slack * 1000)
        if slack < 0: misses += 1

t1, t2 = threading.Thread(target=producer), threading.Thread(target=consumer)
t1.start(); t2.start(); t1.join(); t2.join()

hr = np.array(headroom)
plt.figure(figsize=(8, 2.6))
plt.hist(hr, bins=60)
plt.axvline(0, color="r", linewidth=1.5, label="deadline")
plt.xlabel("headroom at completion [ms]"); plt.legend()
plt.title(f"headroom histogram over {n_blocks} blocks — {misses} deadline misses")
plt.tight_layout(); plt.show()
print(f"block budget {block_dur*1000:.1f} ms | median headroom {np.median(hr):.2f} ms | worst {hr.min():.2f} ms | misses {misses}")
print("engineering rule: if the worst-case headroom is a small fraction of the budget, you WILL glitch")
print("under load — re-run this while the Scale_NN notebook trains to see the OS steal your margin.")

block budget 21.3 ms | median headroom 21.07 ms | worst 20.81 ms | misses 0
engineering rule: if the worst-case headroom is a small fraction of the budget, you WILL glitch
under load — re-run this while the Scale_NN notebook trains to see the OS steal your margin.


/tmp/ipykernel_2703251/1821338709.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Six dB per bit, headroom before shifting, latency ≥ one block, compute < one block-duration, and always look at the *worst-case* headroom, not the average. When the budget can't be met on a CPU, you now know both escape hatches: [GPU batching](../Intro_GPU/README.md) (throughput, at latency cost) and [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) (deterministic latency, at effort cost).

---
## Where next

- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — the same Q15 FIR as literal hardware.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — MHz-rate streams where these budgets get serious.
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — the scheduler that owns your jitter.